# News To Stock Analyser 데이터준비

뉴스기사를 전달하면, 긍/부정분석 뿐아니라, 특정주식에 대한 긍/부정평가 처리 RAG 구현

In [1]:
%pip install -Uqqq datasets langchain-openai


Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN')

os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

### 데이터준비
https://huggingface.co/datasets/daekeun-ml/naver-news-summarization-ko

In [3]:
from datasets import load_dataset

dataset = load_dataset('daekeun-ml/naver-news-summarization-ko')
dataset

README.md:   0%|          | 0.00/4.69k [00:00<?, ?B/s]

c:\Workspace\LLM\llm_venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lsm15\.cache\huggingface\hub\datasets--daekeun-ml--naver-news-summarization-ko. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train.csv: reconstructing file:   0%|          |  0.00B / 66.3MB            

train.csv: downloading bytes:           |  0.00B            

validation.csv:   0%|          | 0.00/7.45M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/8.17M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/22194 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2466 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2740 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 22194
    })
    validation: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2466
    })
    test: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2740
    })
})

In [ ]:
# 필터링 : train / category = 'economy' 인 것만 남김
economy_dataset = dataset['train'].filter(lambda row: row['category'] == 'economy')
print(len(economy_dataset))

Filter:   0%|          | 0/22194 [00:00<?, ? examples/s]

17088


In [6]:
economy_dataset[0]


{'date': '2022-07-03 17:14:37',
 'category': 'economy',
 'press': 'YTN ',
 'title': '추경호 중기 수출지원 총력 무역금융 40조 확대',
 'document': '앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.',
 'link': 'https://n.news.naver.com/mne

In [ ]:
import pandas as pd

df = economy_dataset.to_pandas() # Dataset -> Pandas DataFrame
df.head()

,date,category,press,title,document,link,summary
0,2022-07-03 17:14:37,economy,YTN,추경호 중기 수출지원 총력 무역금융 40조 확대,앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 ...,https://n.news.naver.com/mnews/article/052/000...,"올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, ..."
1,2022-07-04 08:07:12,economy,아시아경제,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다,문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준...,https://n.news.naver.com/mnews/article/277/000...,인터엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타...
2,2022-07-01 08:51:12,economy,뉴시스,에디슨이노 이승훈 제이스페이스 대표 사내이사 선임,기사내용 요약 우주발사체 사업 본격화 서울 뉴시스 김경택 기자 에디슨이노가 우주발사...,https://n.news.naver.com/mnews/article/003/001...,에디슨이노는 1일 임시주주총회를 통해 사명을 이노시스 로 변경하고 이승훈 제이스페이...
3,2022-07-01 16:11:01,economy,머니투데이,SK바사 해외 사업 조직 개편… 글로벌 탑티어 기업으로 성장 박차,SK바이오사이언스가 글로벌 사업의 고도화를 위해 조직 개편을 단행했다. SK바이오사...,https://n.news.naver.com/mnews/article/008/000...,SK바이오사이언스가 글로벌 사업의 고도화를 위해 기존 해외사업개발실을 백신사업뿐만 ...
4,2022-07-01 21:48:04,economy,경향신문,금융당국 “증시 변동성 완화 조치”,4일부터 석달간 증권사 신용융자담보비율 유지의무 면제 금융당국이 코스피지수가 장중 ...,https://n.news.naver.com/mnews/article/032/000...,1일 1일 금융위원회는 증권 유관기관과 금융시장합동점검회의를 열고 코스피지수가 장중...


## sLLM 답변데이터 생성
llm을 이용해서 sLLM이 답변했으면 하는 내용을 생성해낸다. 이때 답변을 품질이 중요하므로, 되도록 상위모델을 사용하는 것이 좋다.

In [14]:
# 분석용 출력 스키마(Pydantic) + 프롬프트 템플릿 구성
from pydantic import BaseModel, Field # 구조화 출력 스키마 정의
from typing import List, Optional # 타입 힌트
from langchain_core.prompts import ChatPromptTemplate # 채팅 프롬프트 템플릿

# 금융뉴스 분석결과 출력 클래스
class StockAnalysis(BaseModel):
    stock_related: bool = Field(description="뉴스와 주식 종목간의 연관성 여부")
    summary: str = Field(description='뉴스 요약')
    # 수혜 종목 리스트 / 근거 키워드 / 이유
    positive_stocks: List[str] = Field(description='긍정적인 영향이 예상되는 주식 종목명 목록')
    positive_keywords: List[str] = Field(description='긍정적인 영향의 근거가 되는 키워드 목록')
    positive_reasons: str = Field(description='긍정적인 영향이 예상되는 이유')
    # 피해 종목 리스트 / 근거 키워드 / 이유
    negative_stocks: List[str] = Field(description='부정적인 영향이 예상되는 주식 종목명 목록')
    negative_keywords: List[str] = Field(description='부정적인 영향의 근거가 되는 키워드 목록')
    negative_reasons: str = Field(description='부정적인 영향이 예상되는 이유')

system_prompt = '''
당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,
특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.

다음 출력지시사항을 지켜주세요.
1. 뉴스와 종목간의 연관성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
2. 뉴스와 종목간의 연관성을 발견했다면:
    - stock_related를 True로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.
    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.
    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.
'''

user_prompt = '''
다음 뉴스기사의 내용에 대해 심층적인 분석을 수행해주세요.

[news]
{news}
'''

prompt = ChatPromptTemplate.from_messages([
    ('system', system_prompt),
    ('human', user_prompt)
])
news = df['document'][0]
prompt.invoke({'news': news})

ChatPromptValue(messages=[SystemMessage(content="\n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n", additional_kwargs={}, response_metadata={}), HumanMessage(content='\n다음 뉴스기사의 내용에 대해 심층적인 분석을 수행해주세요.\n\n[news]\n앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추

In [15]:
# Langchain 구성 및 구조화 출력
from langchain.chat_models import init_chat_model

llm = init_chat_model('gpt-5.6-luna')
chain = prompt | llm.with_structured_output(StockAnalysis) # 프롬프트 -> LLM -> stockAnalysis 구조화 출력
# 뉴스 본문 기반으로 구조화된 분석 결과 반환하는 함수
def analyze_news(news):
    return chain.invoke({'news': news})

analyze_news(news)

StockAnalysis(stock_related=True, summary='정부는 원자재 가격 상승과 해상운임 등 대외 리스크에 대응하고 수출 증가세를 유지하기 위해 하반기 수출 지원책을 강화하기로 했다. 수출 중소·중견기업 대상 무역금융을 연초 계획보다 40조 원 늘린 301조 원으로 확대하고, 물류비 지원과 월 4척 이상의 임시선박 투입, 중소기업 전용 선복 확대 등을 추진한다. 또한 약 2,500개 수출기업의 해외 전시회 참가를 지원하고, 반도체 등 첨단산업 육성 전략과 에너지 효율화 대책을 마련해 수출 경쟁력과 무역수지를 개선할 계획이다.', positive_stocks=['삼성전자', 'SK하이닉스', '현대차', '기아'], positive_keywords=['수출 지원', '무역금융 301조 원', '반도체 등 첨단산업 육성', '물류비 부담 완화', '해외 마케팅 지원'], positive_reasons='무역금융 확대와 물류비·선복 지원은 수출기업의 운전자금 및 물류 부담을 낮춰 수출 물량과 수익성 개선에 기여할 수 있다. 특히 정부가 반도체를 포함한 첨단산업 육성 전략을 추진하기로 한 만큼 글로벌 수출 비중이 높은 삼성전자와 SK하이닉스에 우호적이며, 해외 판매 비중이 높은 현대차와 기아에도 수출 확대 및 공급망 비용 완화 측면에서 긍정적으로 작용할 가능성이 있다. 다만 정책 효과는 실제 금융 집행 속도, 해상운임 추이, 글로벌 수요에 따라 달라질 수 있다.', negative_stocks=[], negative_keywords=[], negative_reasons='')

In [16]:
news = df['document'][100]
display(news) # Jupyter / Colab용 화면출력
print()

analyze_news(news)



'해수부 5일 개정 공유수면 관리 및 매립에 관한 법률 시행 헤럴드경제 홍태화 기자 앞으로 공유수면관리청이 어업·환경 등에 영향을 미칠 것으로 예상되는 공유수면 점용·사용 허가를 할 때 미리 어업인 등 이해관계자들의 의견을 들어야 한다. 해양수산부는 5일 이같은 내용이 담긴 개정 공유수면 관리 및 매립에 관한 법률과 같은 법 시행령·시행규칙이 이날부터 시행된다고 밝혔다. 바다·바닷가·하천 등 공유수면은 공유재이기 때문에 이를 점용·사용하기 위해서는 별도의 허가를 받아야 한다. 최근 해상풍력 발전시설 해변을 이용한 관광시설 등 대규모 시설이 공유수면을 장기적으로 점용·사용하는 경우가 늘어났지만 이해 관계자의 의견을 사전에 수렴할 수 없는 문제가 있었다. 이에 공유수면 점용·사용으로 인한 사회적 갈등이 증가했다. 이러한 문제를 해결하기 위해 해수부는 지난 1월 공유수면 점용·사용 허가를 할 때 이해관계자의 의견을 듣도록 공유수면 관리 및 매립에 관한 법률을 개정했다. 법 개정에 따라 공유수면관리청이 해양환경·수산자원·자연경관 보호 등에 영향을 끼칠 수 있는 공유수면 점용·사용 신청을 받은 경우 이를 관보 공보 와 인터넷 홈페이지에 공고해야 한다. 또 점용·사용 허가를 했을 때 피해를 볼 것으로 예상되는 어업인에 대한 의견 조사도 별도로 진행해야 한다. 황준성 해수부 해양공간정책과장은 공유수면 점용·사용으로 인한 이해 관계자의 피해를 방지하려는 법령 개정의 취지를 달성할 수 있도록 각 공유수면관리청과 협력해 관련 제도의 차질 없는 운영을 지원하겠다 고 말했다.'

StockAnalysis(stock_related=True, summary='해양수산부가 공유수면 점용·사용 허가 과정에서 어업인 등 이해관계자의 의견을 사전에 수렴하도록 개정된 공유수면법과 시행령·시행규칙을 5일부터 시행한다. 해상풍력 발전시설, 해변 관광시설 등 대규모·장기 점용 사업이 증가하면서 발생한 어업권·환경·경관 훼손 관련 갈등을 줄이기 위한 조치다. 해양환경·수산자원·자연경관에 영향을 줄 가능성이 있는 사업은 관보·공보·인터넷 홈페이지에 공고해야 하며, 피해가 예상되는 어업인에 대한 별도 의견조사도 진행된다. 제도 시행으로 공유수면 개발사업의 절차적 투명성과 수용성은 높아질 수 있지만, 사업자 입장에서는 인허가 기간 연장, 추가 협의·보완 비용, 주민·어업인 반대에 따른 사업 지연 가능성이 커질 수 있다. 특히 해상풍력은 공유수면을 장기간 점용하고 어업활동 및 해양환경과 직접 충돌할 가능성이 높아 이번 규제의 영향이 상대적으로 크다.', positive_stocks=[], positive_keywords=[], positive_reasons='', negative_stocks=['SK오션플랜트', '씨에스윈드', '두산에너빌리티'], negative_keywords=['해상풍력 인허가 지연', '이해관계자 의견수렴', '어업권 갈등', '공유수면 점용·사용', '사업 불확실성'], negative_reasons='이번 개정은 해상풍력 발전사업의 허가를 직접 제한하는 조치는 아니지만, 어업인 의견조사와 공고·협의 절차가 추가되면서 신규 해상풍력 프로젝트의 인허가 기간과 개발비가 늘어날 가능성이 있다. 이에 따라 해상풍력 하부구조물·타워·터빈 등 관련 수주를 확보한 기업도 프로젝트 착공 및 발주 시점이 늦어질 경우 매출 인식과 수주 가시성이 약화될 수 있다. SK오션플랜트는 해상풍력 하부구조물, 씨에스윈드는 풍력타워, 두산에너빌리티는 해상풍력 터빈 및 관련 설비에 대한 노출도가 있어 잠재적인 부정 요인이 될 수 있다. 다만 기존에 인허가가 상당 부분

In [18]:
df = df[:10]
df['content'] = df['title'] + '\n' + df['document']

pd.set_option('display.max_colwidth', None)
df['content'].head()

0                                                                                                                                                                      추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 

In [ ]:
# 증류방식
from tqdm.auto import tqdm

results = []

for content in tqdm(df['content']):
    result = analyze_news(content)
    results.append(result)
df['result'] = results
df.head()

In [ ]:
# Pydantic(StockAnalysis) 겨과를 JSON 문자열로 파싱
def parse_to_json(obj):
    return obj.model_dump_json() # StockAnalysis -> JSON 문자열 파싱

df['result_json'] = df['result'].apply(parse_to_json) # df 각 행마다 함수 적용해 파생변수 생성
df.head()

pydantic.BaseModel.model_dump() -> dict  
pydantic.BaseModel.model_dump_json() -> json_str

In [ ]:
# result_json 컬럼 결측치 제거 / 인덱스 재정렬
df = df.dropna(subset=['result_json'])
df = df.reset_index(drop=True)
df.head()

## 학습용 데이터셋 변환
- system
- user(human)
- assistant(ai)

In [ ]:
df['system'] = system_prompt

df = df.rename(columns={
    'content' : 'user',
    'results_json' : 'assistant'
})

df[['system', 'user', 'assistant']]

In [ ]:

df[['system', 'user', 'assistant']].to_json(
    'train.json',
    orient = 'records',
    force_ascii=False,
    indent = 4
)

In [ ]:
import os
from datasets import Dataset

dataset = Dataset.from_pandas(df[['system', 'user', 'assistant']])
dataset.push_to_hub(
    'ca',
    token = os.environ['HF_TOKEN']
)